# 🔎 เปรียบเทียบความเนียนการลบพื้นหลังบนภาพ 4K (u2net vs birefnet-general)
สมุดงานนี้สร้างขึ้นเพื่อทดสอบ "ความละเอียดของขอบภาพ (Pixel-perfect)" โดยเฉพาะ
เราจะนำภาพไปอัปสเกลให้เป็น 4K ก่อน แล้วนำภาพ 4K นั้นมาทดสอบลบพื้นหลังด้วยโมเดล 2 ตัวที่ดีที่สุดในโลก Open Source ตอนนี้:
1. **`u2net`** (มาตรฐาน เร็ว)
2. **`birefnet-general`** (สถาปัตยกรรมใหม่ ความแม่นยำสูงมาก)

### 1. เช็ค GPU และติดตั้งไลบรารี

In [ ]:
!nvidia-smi
!pip install rembg[gpu] realesrgan basicsr opencv-python-headless requests pillow matplotlib

### 2. โหลดภาพและเตรียมภาพ 4K (RealESRGAN)

In [ ]:
import cv2
import numpy as np
import time
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

# โหลดภาพที่มีขน/เส้นผม เพื่อใช้ดูความเนียนของการไดคัท
image_url = 'https://raw.githubusercontent.com/danielgatis/rembg/master/examples/animal-1.jpg'
response = requests.get(image_url)
img_pil = Image.open(BytesIO(response.content)).convert("RGB")

print("🔄 กำลังอัปสเกลเป็น 4K...")
import sys
try:
    import torchvision.transforms.functional_tensor
except ImportError:
    try:
        import torchvision.transforms.functional as functional_tensor
        sys.modules['torchvision.transforms.functional_tensor'] = functional_tensor
    except ImportError:
        pass

from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(
    scale=4,
    model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    model=model,
    half=True,
    gpu_id=0
)
img_cv2 = np.array(img_pil)[:, :, ::-1]
upscaled_cv2, _ = upsampler.enhance(img_cv2, outscale=4)
img_4k_pil = Image.fromarray(upscaled_cv2[:, :, ::-1])

print(f"✅ เตรียมภาพ 4K พร้อมแล้ว! ขนาด: {img_4k_pil.size}")

### 3. ทดสอบลบพื้นหลังด้วยโมเดล 2 แบบ (เปรียบเทียบ)

In [ ]:
from rembg import remove, new_session

results = {}

print("=== 1. ทดสอบรุ่นมาตรฐาน (u2net) ===")
session_u2net = new_session('u2net')

# แบบปิด Alpha Matting
t0 = time.time()
res_u2net_no_alpha = remove(img_4k_pil, session=session_u2net, alpha_matting=False)
t1 = time.time()
print(f"u2net (ไม่ใช้ Alpha Matting) ใช้เวลา: {t1-t0:.2f} วิ")
results['u2net (No Matting)'] = res_u2net_no_alpha

# แบบเปิด Alpha Matting (เพื่อความเนียน)
t0 = time.time()
res_u2net_alpha = remove(
    img_4k_pil, 
    session=session_u2net, 
    alpha_matting=True,
    alpha_matting_foreground_threshold=240,
    alpha_matting_background_threshold=10,
    alpha_matting_erode_size=10
)
t1 = time.time()
print(f"u2net (เปิด Alpha Matting) ใช้เวลา: {t1-t0:.2f} วิ")
results['u2net (With Matting)'] = res_u2net_alpha

print("\n=== 2. ทดสอบรุ่นท็อป (birefnet-general) ===")
session_biref = new_session('birefnet-general')

t0 = time.time()
res_biref_no_alpha = remove(img_4k_pil, session=session_biref, alpha_matting=False)
t1 = time.time()
print(f"birefnet-general (ไม่ใช้ Alpha Matting) ใช้เวลา: {t1-t0:.2f} วิ")
results['birefnet (No Matting)'] = res_biref_no_alpha

t0 = time.time()
res_biref_alpha = remove(
    img_4k_pil, 
    session=session_biref, 
    alpha_matting=True,
    alpha_matting_foreground_threshold=240,
    alpha_matting_background_threshold=10,
    alpha_matting_erode_size=10
)
t1 = time.time()
print(f"birefnet-general (เปิด Alpha Matting) ใช้เวลา: {t1-t0:.2f} วิ")
results['birefnet (With Matting)'] = res_biref_alpha


### 4. ซูมดูรายละเอียดขอบภาพ (Pixel-Peeping)
เราจะทำพื้นหลังหมากรุก และ ครอป (Crop) บริเวณขอบที่มีขน/ผม เพื่อซูมดูพิกเซลเปรียบเทียบกันชัดๆ

In [ ]:
def add_checkerboard(img):
    from PIL import ImageDraw
    bg = Image.new('RGB', img.size, (255, 255, 255))
    draw = ImageDraw.Draw(bg)
    sq_size = 40
    for y in range(0, img.height, sq_size):
        for x in range(0, img.width, sq_size):
            if (x // sq_size + y // sq_size) % 2 == 0:
                draw.rectangle([x, y, x+sq_size, y+sq_size], fill=(200, 200, 200))
    bg.paste(img, (0, 0), img)
    return bg

# กำหนดพิกัดที่จะ Crop ซูมดู (อาจต้องปรับค่า x, y ตามรูปภาพที่ใช้)
# สมมติว่าบริเวณขนหรือหูของสัตว์อยู่แถวๆ นี้
crop_box = (800, 500, 1500, 1200) # (left, upper, right, lower)

fig, axes = plt.subplots(2, 2, figsize=(15, 15))
axes = axes.flatten()

for idx, (name, img_rgba) in enumerate(results.items()):
    # ใส่พื้นหลังหมากรุก
    img_bg = add_checkerboard(img_rgba)
    # ครอปเฉพาะส่วนขอบมาซูมดู
    img_crop = img_bg.crop(crop_box)
    
    axes[idx].imshow(img_crop)
    axes[idx].set_title(name, fontsize=16, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()